# Paso 1: Importacion de librerias

**Que hace:** Importa todas las librerias necesarias para analisis, preprocesamiento, modelado y evaluacion de la red neuronal.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

np.random.seed(42)
tf.random.set_seed(42)

In [ ]:
# Paso 2: Carga del dataset

df = pd.read_csv("Breast_cancer_Reseach.csv")
df.head()

,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave_points_mean,symmetry_mean,fractal_dimension_mean,...,perimeter_se,area_se,smoothness_se,radius_worst,texture_worst,perimeter_worst,area_worst,concavity_worst,concave_points_worst,diagnosis
0,15.490142,19.500898,75.565249,526.409677,0.110140,0.094296,0.211172,0.109355,0.114075,0.054322,...,2.462022,42.305337,0.011282,18.287341,22.645848,101.643449,980.949501,0.300743,0.116431,B
1,13.585207,17.282378,93.536417,542.133582,0.098852,0.118453,0.205202,0.146600,0.218061,0.065125,...,1.592161,32.253159,0.007029,13.874569,27.706897,120.003792,734.421427,0.292429,0.114310,B
2,15.943066,19.489190,79.066398,957.471814,0.072091,0.102897,0.265062,0.125013,0.185707,0.062901,...,3.109572,61.427110,0.007098,19.765426,23.100544,165.333024,865.634991,0.138234,0.221270,B
3,18.569090,21.173192,84.566898,650.102423,0.085842,0.122600,0.132492,0.093229,0.168611,0.060439,...,3.020457,54.395979,0.005667,14.638553,25.579304,101.911156,808.195350,0.218719,0.243221,M
4,13.297540,19.195440,123.469042,767.319500,0.094745,0.139292,0.060506,0.154095,0.205715,0.069305,...,2.027102,32.581794,0.007993,19.998220,25.872578,89.016818,987.071625,0.520519,0.103556,M


In [ ]:
# Paso 3: Exploracion inicial de datos

print("Forma del dataset:", df.shape)
print("\nTipos de datos:")
print(df.dtypes)

print("\nValores nulos por columna:")
print(df.isnull().sum())

radius_mean               0
texture_mean              0
perimeter_mean            0
area_mean                 0
smoothness_mean           0
compactness_mean          0
concavity_mean            0
concave_points_mean       0
symmetry_mean             0
fractal_dimension_mean    0
radius_se                 0
texture_se                0
perimeter_se              0
area_se                   0
smoothness_se             0
radius_worst              0
texture_worst             0
perimeter_worst           0
area_worst                0
concavity_worst           0
concave_points_worst      0
diagnosis                 0
dtype: int64

# Paso 4: Analisis de la variable objetivo

**Que hace:** Revisa la distribucion de la columna diagnosis para identificar el balance entre tumores benignos y malignos.

In [ ]:
# Conteo de clases en la variable objetivo

plt.figure(figsize=(6, 4))
sns.countplot(data=df, x="diagnosis", hue="diagnosis", palette="Set2", legend=False)
plt.title("Distribucion de diagnosticos")
plt.xlabel("Diagnosis (B=Benigno, M=Maligno)")
plt.ylabel("Cantidad")
plt.show()

print(df["diagnosis"].value_counts())

# Paso 5: Preprocesamiento de datos

**Que hace:** Convierte la variable diagnosis a formato numerico (B=0, M=1) y separa variables predictoras (X) y objetivo (y).

In [ ]:
# Conversion de variable objetivo y separacion de datos

df["diagnosis"] = df["diagnosis"].map({"B": 0, "M": 1})

X = df.drop(columns=["diagnosis"])
y = df["diagnosis"]

print("Dimension de X:", X.shape)
print("Dimension de y:", y.shape)
print("\nDistribucion de y:")
print(y.value_counts())

# Paso 6: Division del dataset

**Que hace:** Divide el dataset en entrenamiento (80%) y prueba (20%), conservando la proporcion de clases con stratify.

In [ ]:
# Split de datos con estratificacion

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

# Paso 7: Escalado de datos

**Que hace:** Estandariza las variables con StandardScaler para que la red neuronal entrene de forma estable.

In [ ]:
# Estandarizacion de caracteristicas

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Matriz escalada de entrenamiento:", X_train_scaled.shape)
print("Matriz escalada de prueba:", X_test_scaled.shape)

# Paso 8: Construccion de la red neuronal

**Que hace:** Define una arquitectura simple y efectiva para clasificacion binaria, evitando un modelo demasiado complejo.

In [ ]:
# Arquitectura del modelo

model = Sequential([
    Dense(32, activation="relu", input_shape=(X_train_scaled.shape[1],)),
    Dropout(0.2),
    Dense(16, activation="relu"),
    Dense(1, activation="sigmoid"),
])

model.summary()

# Paso 9: Compilacion del modelo

**Que hace:** Configura la red con Adam, perdida binary_crossentropy y metrica accuracy para clasificacion binaria.

In [ ]:
# Compilacion

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"],
)